#Downloading the dataset and Imports (T5 Model Traditional)


In [ ]:
# 1. Upload your Kaggle API token (kaggle.json)
from google.colab import files
print("▶ Upload your kaggle.json (from your Kaggle account settings)")
files.upload()   # select your kaggle.json

# 2. Configure Kaggle CLI
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# 3. Download & unzip the dataset
!mkdir -p data
!kaggle datasets download -d sunnysai12345/news-summary -p data/
!unzip -o data/news-summary.zip   -d data/

▶ Upload your kaggle.json (from your Kaggle account settings)


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/sunnysai12345/news-summary
License(s): GPL-2.0
Archive:  data/news-summary.zip
  inflating: data/news_summary.csv   
  inflating: data/news_summary_more.csv  


In [ ]:
!pip install -q kaggle transformers datasets pandas

In [ ]:
!pip install -q evaluate

In [ ]:
import pandas as pd
import numpy as np
from datasets import Dataset, load_dataset
from evaluate import load
from transformers import (
    T5TokenizerFast, T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer
)

RuntimeError: operator torchvision::nms does not exist

Load Dataset and split it 90%-10%

In [ ]:
# 5.1 Load CSV
# Kaggle file is named "news_summary.csv" after unzipping
df = pd.read_csv('data/news_summary.csv', encoding='latin-1')
df = df.rename(columns={'headlines':'summary'})

# 5.2 Split train/test
ds = Dataset.from_pandas(df)
ds = ds.train_test_split(test_size=0.1, seed=42)



Model and tokenizer built in functions

In [ ]:
# 5.3 Tokenizer & model
model_name = 't5-small'   # swap for t5-base, bart-base, etc.
tokenizer  = T5TokenizerFast.from_pretrained(model_name)
model      = T5ForConditionalGeneration.from_pretrained(model_name)

# 5.4 Preprocessing
max_input_length  = 512
max_target_length = 64

def preprocess(batch):
    inputs = tokenizer(
        ["summarize: " + t for t in batch['text']],
        max_length=max_input_length,
        truncation=True
    )
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            batch['summary'],
            max_length=max_target_length,
            truncation=True
        )
    inputs['labels'] = labels['input_ids']
    return inputs

tokenized = ds.map(
    preprocess,
    batched=True,
    remove_columns=['text','summary']
)

# 5.5 Data collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)




Map:   0%|          | 0/4062 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3980: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/452 [00:00<?, ? examples/s]

Implementation of Training args, the evalution metric we used rouge, and Calling the pipeline

In [ ]:
!pip install -q --upgrade transformers

In [ ]:
!pip install -q rouge_score


In [ ]:
# 5.6 Training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir='./t5-news-summary',
    do_train=True,
    do_eval=True,
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    logging_steps=200,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
)

rouge = load('rouge')

def safe_decode(sequences, tokenizer):
    """
    Clips out-of-range token IDs, maps -100→pad_token_id,
    and casts everything to plain Python ints before decoding.
    """
    decoded = []
    vocab_size   = tokenizer.vocab_size
    pad_token_id = tokenizer.pad_token_id

    for seq in sequences:
        clean_ids = []
        for tok in seq:
            t = int(tok)
            if t < 0 or t >= vocab_size:
                clean_ids.append(pad_token_id)
            else:
                clean_ids.append(t)
        decoded.append(tokenizer.decode(clean_ids, skip_special_tokens=True))
    return decoded

def compute_metrics(pred):
    # 1) Get raw predictions & labels
    preds     = pred.predictions
    label_ids = pred.label_ids

    # 2) Map -100 → pad_token_id in labels
    label_ids = np.where(label_ids != -100,
                         label_ids,
                         tokenizer.pad_token_id)

    # 3) Decode both sets of sequences
    decoded_preds  = safe_decode(preds,     tokenizer)
    decoded_labels = safe_decode(label_ids, tokenizer)

    # 4) Compute ROUGE (returns plain floats)
    result = rouge.compute(predictions=decoded_preds,
                           references=decoded_labels)

    # 5) Convert to %-scores
    return {k: v * 100 for k, v in result.items()}




In [ ]:
# 5.8 Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized['train'],
    eval_dataset= tokenized['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)


<ipython-input-5-56185b64947e>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


In [ ]:
# 5.9 Train!
trainer.train()



wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: amrwael-elwakil (amrwael-elwakil-msa-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss
200,2.504600
400,2.118200
600,2.026000
800,1.916500
1000,1.903900
1200,1.820200
1400,1.861700


TrainOutput(global_step=1524, training_loss=2.0040508080029427, metrics={'train_runtime': 130.3339, 'train_samples_per_second': 93.498, 'train_steps_per_second': 11.693, 'total_flos': 341826338095104.0, 'train_loss': 2.0040508080029427, 'epoch': 3.0})

Evaluation

In [ ]:
import numpy as np
import torch

# 1) Use trainer.predict on your test split
pred_out    = trainer.predict(tokenized['test'])
pred_seqs   = pred_out.predictions    # (num_examples, seq_len)
label_seqs  = pred_out.label_ids      # (num_examples, seq_len)

# 2) Clean up label_ids as above
label_seqs = np.where(label_seqs != -100,
                      label_seqs,
                      tokenizer.pad_token_id)

# 3) Decode everything
decoded_preds  = safe_decode(pred_seqs,  tokenizer)
decoded_labels = safe_decode(label_seqs, tokenizer)

# 4) Compute ROUGE
rouge_scores = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels
)

# 5) Print nicely
print({k: f"{v*100:.2f}" if v < 1 else f"{v:.2f}" for k,v in rouge_scores.items()})







{'rouge1': '47.93', 'rouge2': '26.00', 'rougeL': '43.87', 'rougeLsum': '43.84'}


In [ ]:
bertscore = load("bertscore")
bs = bertscore.compute(predictions=decoded_preds,
                       references=decoded_labels,
                       model_type="roberta-large")
f1_scores = bs["f1"]
avg_f1     = sum(f1_scores) / len(f1_scores) * 100
print("BERTScore-F1:", avg_f1)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BERTScore-F1: 90.51429580534453


In [ ]:
bleu   = load("bleu")
meteor = load("meteor")

print("BLEU:",   bleu.compute(predictions=decoded_preds,
                              references=[[r] for r in decoded_labels])["bleu"])
print("METEOR:", meteor.compute(predictions=decoded_preds,
                                 references=decoded_labels)["meteor"])

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


BLEU: 0.1838167890564692
METEOR: 0.43912550214912444


In [ ]:
# 1) Figure out which device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2) Move the model there
model = model.to(device)

# 3) Tokenize as before, then move the tensors
sample = """Last month, the small coastal town of Harborview was rocked by an unexpected discovery beneath its tranquil waters. Local fisherman Maria Delgado was the first to notice something unusual while casting her nets one morning: a series of gleaming, metallic objects protruding from the sandy seabed. Word spread quickly, and within hours marine biologists from the nearby Oceanic Research Institute were wading in to investigate.

Dr. Samuel Hayes, lead archaeologist on the dive team, explained they had uncovered the remnants of what appears to be a 19th-century merchant vessel. Barnacle-encrusted hull fragments, a corroded brass compass, and dozens of intact ceramic jugs bearing the crest of a long-defunct trading company were carefully documented and raised to the surface. Tests show the ship likely sank during a November gale in 1878, carrying goods bound for European ports.

The find has excited both historians and local residents, who are planning a small museum exhibit for the recovered artifacts. Mayor Elaine Ford announced, “This discovery ties our community directly to an important chapter of maritime commerce. We’ll work with the institute to preserve these relics and share our town’s story with the world.” Visitors can expect a temporary display to open by late summer, with digital 3D reconstructions so people around the globe can explore the sunken vessel virtually.

Meanwhile, the research team continues careful excavation, hopeful that further exploration will yield personal items—letters, coins, even navigational charts—that bring to life the human stories aboard that long-forgotten ship."""
encoded = tokenizer(
    "summarize in 2–3 sentences: " + sample,
    return_tensors='pt',
    truncation=True,
    max_length=512
)
input_ids = encoded.input_ids.to(device)
attention_mask = encoded.attention_mask.to(device)

# Generate a longer summary
outputs = model.generate(
    input_ids,
    attention_mask=attention_mask,
    max_length=150,        # allow up to ~150 tokens
    min_length=60,         # require at least ~60 tokens
    num_beams=4,           # beam search for better quality
    length_penalty=1.2,    # encourage longer outputs
    no_repeat_ngram_size=3,# avoid verbatim repetition
    early_stopping=True
)

long_summary = tokenizer.decode(outputs[0].cpu(), skip_special_tokens=True)
print("\n▶︎ LONGER SUMMARY:\n", long_summary)


▶︎ LONGER SUMMARY:
 Harborview rocked by gleaming, metallic objects in a day: Scientists. ship likely sank during gale in 1878, carrying goods bound for European ports. museum to open by late summer to explore ship's relics: Mayor


#TF-IDF TF-IDF Sentence Scoring Summarizer


In [ ]:
pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 132.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 106.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstal

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from evaluate import load
import pandas as pd
from datasets import Dataset



In [ ]:
df = pd.read_csv('data/news_summary.csv', encoding='latin-1')
ds = Dataset.from_pandas(df)
split = ds.train_test_split(test_size=0.1, seed=42)
test_ds    = split['test']
test_texts = test_ds['text']
test_refs  = test_ds['headlines']


In [ ]:
# 2) Evaluation helper (reuse from before)
def eval_summaries(preds, refs):
    # filter out empty refs
    pairs = [(p,r) for p,r in zip(preds, refs) if isinstance(r,str) and r.strip()]
    preds_f, refs_f = zip(*pairs)
    rouge     = load("rouge")
    bertscore = load("bertscore")
    bleu      = load("bleu")
    # ROUGE
    r  = rouge.compute(predictions=preds_f, references=refs_f)
    r  = {k: v*100 for k,v in r.items()}
    # BERTScore-F1
    bs = bertscore.compute(predictions=preds_f, references=refs_f, model_type="roberta-large")
    f1 = np.mean(bs["f1"])*100
    # BLEU
    b  = bleu.compute(predictions=preds_f, references=[[r] for r in refs_f])["bleu"]*100
    return r, f1, b

# 3) Build TF-IDF on all test split sentences


In [ ]:
all_sentences = []
indices       = []  # map back to which article & sentence idx
for art_idx, art in enumerate(test_texts):
    sents = [s.strip() for s in art.split('.') if s.strip()]
    for si, s in enumerate(sents):
        all_sentences.append(s)
        indices.append((art_idx, si))

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(all_sentences)



In [ ]:
# 4) Score & pick top-3 sentences per article
tfidf_summaries = [''] * len(test_texts)
for art_idx in range(len(test_texts)):
    # get rows for this article
    rows = [i for i,(a,i_) in enumerate(indices) if a==art_idx]
    if not rows:
        continue
    submat = tfidf_matrix[rows]
    # score each sentence by sum of its tfidf weights
    scores = submat.sum(axis=1).A1
    # pick top 3 sentences (by score), sort back in original order
    topk   = np.argsort(-scores)[:3]
    chosen = sorted([rows[i] for i in topk], key=lambda r: indices[r][1])
    summary = '. '.join(all_sentences[r] for r in chosen) + '.'
    tfidf_summaries[art_idx] = summary



In [ ]:
# 5) Evaluate
r_tfidf, f1_tfidf, b_tfidf = eval_summaries(tfidf_summaries, test_refs)



tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 6) Report
print("=== TF-IDF Extractive (3 sent) ===")
print(f"ROUGE-1: {r_tfidf['rouge1']:.2f}%  ROUGE-2: {r_tfidf['rouge2']:.2f}%  ROUGE-L: {r_tfidf['rougeL']:.2f}%")
print(f"BERTScore-F1: {f1_tfidf:.2f}%  BLEU: {b_tfidf:.2f}%")

=== TF-IDF Extractive (3 sent) ===
ROUGE-1: 21.43%  ROUGE-2: 8.97%  ROUGE-L: 18.23%
BERTScore-F1: 87.63%  BLEU: 2.79%


#LLM


In [ ]:
!pip install -q --upgrade bitsandbytes transformers accelerate evaluate rouge_score bert-score



  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 96.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 102.0 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from evaluate import load

In [ ]:
df = pd.read_csv('data/news_summary.csv', encoding='latin-1')
ds = Dataset.from_pandas(df)
split = ds.train_test_split(test_size=0.1, seed=42)
test_ds    = split['test']
test_texts = test_ds['text']
test_refs  = test_ds['headlines']


In [ ]:
model_id  = "tiiuae/falcon-7b-instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
model     = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)
device = next(model.parameters()).device

tokenizer_config.json:   0%|          | 0.00/1.13k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.73M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/281 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/17.7k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.48G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

In [ ]:
if tokenizer.pad_token is None:
    # Point the pad token to end-of-sequence token
    tokenizer.padding_side = "left"
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id


batch_size = 8
pred_summaries = []

for i in range(0, len(test_texts), batch_size):
    batch = test_texts[i : i + batch_size]
    prompts = [
        "Summarize the following news article in 2–3 concise sentences:\n\n" + art
        for art in batch
    ]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to(device)

    outs = model.generate(
        **inputs,
        max_new_tokens=20,
        length_penalty=0.8,
        no_repeat_ngram_size=2,
        num_beams=4,
        early_stopping=True
    )

    decoded = tokenizer.batch_decode(outs, skip_special_tokens=True)
    for j, summary in enumerate(decoded):
        # strip off the prompt prefix
        pred_summaries.append(summary[len(prompts[j]):].strip())

Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:11 for open-end generation.
Setting `pad_token_id` to `eos_tok

In [ ]:
rouge     = load("rouge")
bertscore = load("bertscore")
bleu      = load("bleu")

In [ ]:
pairs = [
    (p, r)
    for p, r in zip(pred_summaries, test_refs)
    if isinstance(r, str) and r.strip()       # keep only when r is non-blank
]

print(f"Filtered out {len(test_refs) - len(pairs)} empty-ref examples.")

# 2) Unzip back into two lists
preds_valid, refs_valid = zip(*pairs)


Filtered out 0 empty-ref examples.


In [ ]:

# 3) Recompute all metrics on the cleaned lists:
import numpy as np
from evaluate import load

# ROUGE
rouge = load("rouge")
r = rouge.compute(predictions=preds_valid, references=refs_valid)
rouge_scores = {k: v * 100 for k, v in r.items()}

# BERTScore
bertscore = load("bertscore")
bs = bertscore.compute(
    predictions=preds_valid,
    references=refs_valid,
    model_type="roberta-large"
)
bert_f1 = np.mean(bs["f1"]) * 100

# BLEU
bleu = load("bleu")
bleu_score = bleu.compute(
    predictions=preds_valid,
    references=[[r] for r in refs_valid]
)["bleu"] * 100

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print("Falcon-7B-Instruct (FP16 on L4) Evaluation:")
print(f"ROUGE-1     : {rouge_scores['rouge1']:.2f}%")
print(f"ROUGE-2     : {rouge_scores['rouge2']:.2f}%")
print(f"ROUGE-L     : {rouge_scores['rougeL']:.2f}%")
print(f"ROUGE-Lsum  : {rouge_scores['rougeLsum']:.2f}%")
print(f"BERTScore-F1: {bert_f1:.2f}%")
print(f"BLEU        : {bleu_score:.2f}%")

Falcon-7B-Instruct (FP16 on L4) Evaluation:
ROUGE-1     : 14.94%
ROUGE-2     : 1.93%
ROUGE-L     : 12.29%
ROUGE-Lsum  : 12.45%
BERTScore-F1: 83.61%
BLEU        : 0.45%


#bart-large-cnn

In [ ]:
from transformers import pipeline

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=0,
    min_length=5,
    max_length=30
)

preds = [summarizer(text)[0]["summary_text"] for text in test_texts]
# then re-compute your metrics…


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
rouge     = load("rouge")
bertscore = load("bertscore")
bleu      = load("bleu")

In [ ]:
pairs = [(p, r) for p, r in zip(preds, test_refs) if isinstance(r, str) and r.strip()]
preds_f, refs_f = zip(*pairs)

# 5) Compute ROUGE (as %)
r = rouge.compute(predictions=preds_f, references=refs_f)
rouge_scores = {k: v * 100 for k, v in r.items()}

# 6) Compute BERTScore-F1 (average ×100)
bs = bertscore.compute(predictions=preds_f, references=refs_f, model_type="roberta-large")
bert_f1 = np.mean(bs["f1"]) * 100

# 7) Compute BLEU (×100)
bleu_score = bleu.compute(
    predictions=preds_f,
    references=[[ref] for ref in refs_f]
)["bleu"] * 100


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print("BART-large-cnn Evaluation:")
print(f"ROUGE-1     : {rouge_scores['rouge1']:.2f}%")
print(f"ROUGE-2     : {rouge_scores['rouge2']:.2f}%")
print(f"ROUGE-L     : {rouge_scores['rougeL']:.2f}%")
print(f"ROUGE-Lsum  : {rouge_scores['rougeLsum']:.2f}%")
print(f"BERTScore-F1: {bert_f1:.2f}%")
print(f"BLEU        : {bleu_score:.2f}%")

BART-large-cnn Evaluation:
ROUGE-1     : 32.70%
ROUGE-2     : 13.45%
ROUGE-L     : 27.84%
ROUGE-Lsum  : 27.84%
BERTScore-F1: 88.10%
BLEU        : 5.45%


#FINETUNING


In [ ]:
!pip install -q transformers datasets accelerate


In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

RuntimeError: Failed to import transformers.trainer_seq2seq because of the following error (look up to see its traceback):
Failed to import transformers.integrations.integration_utils because of the following error (look up to see its traceback):
Failed to import transformers.modeling_utils because of the following error (look up to see its traceback):
partially initialized module 'torchvision' has no attribute 'extension' (most likely due to a circular import)

In [ ]:
df = pd.read_csv('data/news_summary.csv', encoding='latin-1')
ds = Dataset.from_pandas(df).train_test_split(test_size=0.1, seed=42)
train_ds = ds['train']
test_ds  = ds['test']

In [ ]:
train_ds = train_ds.rename_column('headlines','summary')
test_ds  = test_ds.rename_column('headlines','summary')
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in ['text','summary']])
test_ds  = test_ds.remove_columns([c for c in test_ds.column_names  if c not in ['text','summary']])


In [ ]:
bart_id   = "facebook/bart-large-cnn"
bart_tok  = AutoTokenizer.from_pretrained(bart_id, use_fast=True)
bart_model= AutoModelForSeq2SeqLM.from_pretrained(bart_id)


config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
def preprocess_bart(examples):
    inputs = ["summarize: " + t for t in examples['text']]
    model_inputs = bart_tok(inputs, max_length=512, truncation=True)
    with bart_tok.as_target_tokenizer():
        labels = bart_tok(examples['summary'], max_length=64, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs


In [ ]:
tok_train = train_ds.map(preprocess_bart, batched=True, remove_columns=train_ds.column_names)
tok_test  = test_ds.map(preprocess_bart,  batched=True, remove_columns=test_ds.column_names)

NameError: name 'preprocess_bart' is not defined

In [ ]:
data_collator_bart = DataCollatorForSeq2Seq(bart_tok, model=bart_model)
bart_args = Seq2SeqTrainingArguments(
    output_dir='./bart-ft',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    logging_steps=100,
    save_steps=500,
    save_total_limit=2,
    predict_with_generate=True,    # needed so generate() is used under the hood
)

In [ ]:
bart_trainer = Seq2SeqTrainer(
    model=bart_model,
    args=bart_args,
    train_dataset=tok_train,
    eval_dataset=tok_test,         # pass it here for later .evaluate()
    tokenizer=bart_tok,
    data_collator=data_collator_bart
)

<ipython-input-14-f4ba0438f857>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  bart_trainer = Seq2SeqTrainer(


In [ ]:
bart_trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amrwael-elwakil (amrwael-elwakil-msa-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
100,1.603100
200,1.567400
300,1.506800
400,1.475300
500,1.445900
600,1.427500
700,1.455100
800,1.347200
900,1.396600
1000,1.334200


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:3339: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 142, 'min_length': 56, 'early_stopping': True, 'num_beams': 4, 'length_penalty': 2.0, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


TrainOutput(global_step=3048, training_loss=0.7990589539210001, metrics={'train_runtime': 867.485, 'train_samples_per_second': 14.048, 'train_steps_per_second': 3.514, 'total_flos': 2344688665952256.0, 'train_loss': 0.7990589539210001, 'epoch': 3.0})

In [ ]:
metrics = bart_trainer.evaluate()   # will use test_dataset
print({k: f"{v:.2f}" for k,v in metrics.items()})

{'eval_loss': '1.94', 'eval_runtime': '3.67', 'eval_samples_per_second': '123.03', 'eval_steps_per_second': '30.76', 'epoch': '3.00'}


In [ ]:
import pandas as pd

# adjust the path to wherever your CSV actually lives
df = pd.read_csv('data/news_summary.csv', encoding='latin-1')
test_df    = df.sample(frac=0.1, random_state=42)            # same split you used before
test_texts = test_df['text'].tolist()
test_refs  = test_df['headlines'].tolist()

In [ ]:
from transformers import pipeline


In [ ]:
bart_pipe = pipeline(
    "summarization",
    model=bart_model,
    tokenizer=bart_tok,
    device=0,
    min_length=5, max_length=30, num_beams=4
)
bart_preds = [bart_pipe(t)[0]["summary_text"] for t in test_texts]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


In [ ]:
!pip install -q rouge_score nltk bert_score

from rouge_score import rouge_scorer
from bert_score import score as bert_score
import nltk
from nltk.translate.bleu_score import corpus_bleu
import numpy as np

# 1) Initialize scorers
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def compute_metrics_fallback(preds, refs):
    # Filter out empty refs
    pairs = [(p, r) for p, r in zip(preds, refs) if isinstance(r, str) and r.strip()]
    preds_f, refs_f = zip(*pairs)

    # ROUGE
    r1, r2, rL = 0.0, 0.0, 0.0
    for p, r in zip(preds_f, refs_f):
        scores = scorer.score(r, p)
        r1 += scores['rouge1'].fmeasure
        r2 += scores['rouge2'].fmeasure
        rL += scores['rougeL'].fmeasure
    n = len(preds_f)
    rouge_scores = {
        "ROUGE-1": (r1/n)*100,
        "ROUGE-2": (r2/n)*100,
        "ROUGE-L": (rL/n)*100
    }

    # BLEU (corpus-level)
    # refs for nltk: list of lists of tokens
    ref_tokens = [[ref.split()] for ref in refs_f]
    pred_tokens = [pred.split() for pred in preds_f]
    bleu = corpus_bleu(ref_tokens, pred_tokens) * 100

    # BERTScore
    P, R, F1 = bert_score(preds_f, refs_f, model_type='roberta-large', verbose=False)
    bert_f1 = F1.mean().item() * 100

    metrics = {
        **rouge_scores,
        "BLEU"        : bleu,
        "BERTScore-F1": bert_f1
    }
    return metrics

In [ ]:
bart_metrics   = compute_metrics_fallback(bart_preds,   test_refs)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
print("BART metrics:", bart_metrics)

BART metrics: {'ROUGE-1': 89.26496572135467, 'ROUGE-2': 82.30507528747397, 'ROUGE-L': 87.77902990775405, 'BLEU': 78.58231740081175, 'BERTScore-F1': 97.95008301734924}


FALCON